# Automação da Atualização do DCS

### Desafio:

Assim que resiquitado, o robo deverá atualizar o sistema DCS com as datas de faturamento fornecida pelo departamento de finanças.

In [2]:
import pandas as pd
import pyautogui
import time
from datetime import datetime
import pyperclip 
import os

# pyautogui.click -> clicar com o mouse
# pyautogui.write - > escrever um Texto
# pyautogui.press - > aperta 1 tecla
# pyautogui.hotkey - > combinação de teclas
pyautogui.PAUSE = 1
# Passo a passo do desafio
# Passo 1: Entrar no sistema da empresa (link di drive)
# abrir o Chrome
pyautogui.press("win")
pyautogui.write("chrome")
pyautogui.press("enter")
time.sleep(2)
pyautogui.click(x=727, y=582) # Seleciona a conta do google chrome
time.sleep(2)

# digitei o link do dcs
link = "https://dcs.smil.com/global-vehicle/login"
pyautogui.write(link)
# apertei enter e esperei
pyautogui.press("enter")
time.sleep(3)
# fiz o login no sistema dcs
pyautogui.click(x=1365, y=661) # clica em login
time.sleep(3)

# Navegar no menu do sistema DCS

In [3]:
pyautogui.click(x=340, y=424) # clica em invoice para expandir o modulo
pyautogui.click(x=320, y=469) # clica em Sales invoice para expandir o modulo
pyautogui.click(x=225, y=589) # clica em Invoice Creation-by VIN

# Atualizar o chassi no DCS

In [4]:
# --- 1. Carregar e Tratar a Base de Dados ---
caminho = r"C:\Users\fsywo\OneDrive\Área de Trabalho\Faturados.xlsx"
tabela = pd.read_excel(caminho, engine='openpyxl')

# CORREÇÃO DA DATA: Formata a coluna inteira para o padrão Ano-Mês-Dia, sem a hora
tabela["data de faturamento"] = pd.to_datetime(tabela["data de faturamento"]).dt.strftime('%Y-%m-%d')

quantidade_de_registros = len(tabela)
print(f"Total de linhas a serem processadas: {quantidade_de_registros}")

# Captura a data e hora atual do computador
agora = datetime.now()

# Formata para ficar num padrão de fácil leitura (Dia/Mês/Ano Hora:Minuto:Segundo)
data_hora_formatada = agora.strftime("%d/%m/%Y %H:%M:%S")

# Exibe a mensagem que você pediu
print(f"Update Date: {data_hora_formatada}")

# Tempo para o robô começar
time.sleep(3) 

# Cria a variável vazia para guardar os dados do e-mail
relatorio_chassis = ""

# --- 2. Loop de Automação ---
for i in range(quantidade_de_registros):
    
    # Pega os dados direto do Pandas
    chassi = str(tabela.loc[i, "Número do chassi"]) 
    data_fat = str(tabela.loc[i, "data de faturamento"])
    relatorio_chassis += f"• Chassi: {chassi} | Data Atualizada: {data_fat}\n"
    print(f"Processando {i + 1}/{quantidade_de_registros} - Chassi: {chassi} | Data: {data_fat}")

    # --- Pesquisar no sistema DCS o VIN ---
    pyautogui.click(x=459, y=435) # clica em VIN no DCS
    pyperclip.copy(chassi)        # Copia o chassi
    pyautogui.hotkey("ctrl","v")  # Cola no DCS
    pyautogui.click(x=1718, y=421) # clica em Query no DCS
    time.sleep(3)
    
    # --- Editar Chassi no DCS ---
    pyautogui.click(x=422, y=637) # clica na caixa de seleção do VIN
    pyautogui.click(x=474, y=572) # clica em Create by VIN
    time.sleep(1)
    pyautogui.click(x=1079, y=443) # clica em discount type
    pyautogui.click(x=1007, y=504) # clica em No
    pyautogui.press("tab") # ir para o campo de data
    
    # --- Inserir a Data de Faturamento ---
    pyperclip.copy(data_fat)      # Copia a data (já formatada)
    pyautogui.hotkey("ctrl", "v") # Cola no DCS
    
    # --- Salvar ---
    pyautogui.press("tab")
    pyautogui.press("tab")
    pyautogui.click(x=1693, y=591) # Clica em salvar para salvar a linha
    time.sleep(1)
    pyautogui.click(x=1712, y=936) # Clica em salvar para salvar o registro
    
    # --- Apagar o VIN da consulta anterior ---
    time.sleep(2)
    pyautogui.click(x=509, y=425, clicks=2) # Seleciona o VIN da consulta
    pyautogui.press("delete") # Limpa o campo
    
    time.sleep(2) # Pausa antes do próximo ciclo


Total de linhas a serem processadas: 5
Update Date: 03/03/2026 16:55:41
Processando 1/5 - Chassi: LSJWS4097TZ507485 | Data: 2026-03-02
Processando 2/5 - Chassi: LSJWH4092TN013590 | Data: 2026-03-02
Processando 3/5 - Chassi: LSJWH4094TN013588 | Data: 2026-03-02
Processando 4/5 - Chassi: LSJWH4090TN014060 | Data: 2026-03-02
Processando 5/5 - Chassi: LSJWH4097TN014086 | Data: 2026-03-02


# Contador de Lote

In [5]:
# --- LÓGICA DO CONTADOR DE BATCH ---
ficheiro_contador = r"C:\Users\fsywo\OneDrive\Área de Trabalho\contador_batch.txt"

# Verifica se o ficheiro já existe
if os.path.exists(ficheiro_contador):
    # Se existe, lê o número que lá está guardado
    with open(ficheiro_contador, "r") as f:
        conteudo = f.read().strip()
        # Previne erros caso o ficheiro esteja vazio
        lote_atual = int(conteudo) if conteudo.isdigit() else 42
else:
    # Se é a primeira vez que corre, começa no 42 (para somar 1 e virar 43)
    lote_atual = 42 

# Adiciona 1 para a execução atual
lote_atual += 1

# Guarda o novo número de volta no ficheiro para a próxima vez
with open(ficheiro_contador, "w") as f:
    f.write(str(lote_atual))

# Cria a variável com o título final
titulo_email = f"DCS Update - Batch {lote_atual}"
print(f"O título do e-mail será: {titulo_email}")

O título do e-mail será: DCS Update - Batch 45


# Enviar o email com os chassis atualizados

In [6]:
# Abrir o outlook
pyautogui.press("win")
pyautogui.write("Outlook")
pyautogui.press("enter")
time.sleep(5)
pyautogui.click(x=165, y=127) # Seleciona novo email
pyautogui.write("eduardo.fujita@smil.com") # Escreve o email do destinatario 1
time.sleep(1)
pyautogui.press("tab") # Confirma o email
time.sleep(1)
pyautogui.write("carlos.expedito@smil.com") # Escreve o email do destinatario 1
time.sleep(1)
pyautogui.press("tab") # Confirma o email
time.sleep(1)
pyautogui.press("tab") # Troca para o campo CC
pyautogui.press("tab") # Troca para o campo titulo de email
# AQUI ESTÁ A MUDANÇA: Usamos a variável que contém o número do Batch atualizado
pyperclip.copy(titulo_email) 
pyautogui.hotkey("ctrl", "v") 
pyautogui.press("tab") # Confirma o titulo do email e desce para o corpo

# Montando o Corpo do e-mail com a lista de chassis atualizados
texto = f"""Dears,

Follow the DCS Update Report:

Update Date: {data_hora_formatada}

Updated Records:
{relatorio_chassis}

Regards,
"""

# Copia e cola o corpo do e-mail
pyperclip.copy(texto)
pyautogui.hotkey("ctrl", "v") 

time.sleep(1)
# Clica no botão de enviar
pyautogui.click(x=836, y=189) 

print("E-mail preenchido e enviado com sucesso!")

E-mail preenchido e enviado com sucesso!
